# Fonds de défaut — Préparation des membres compensateurs

Ce notebook prépare l'implémentation du fonds de défaut selon le standard **Cover-2** décrit dans `Note_fonds_de_defaut_QF.pdf`.

La première étape consiste à représenter plusieurs membres compensateurs, chacun avec son propre portefeuille. Les mêmes scénarios de stress seront ensuite appliqués à tous les membres afin de calculer leurs pertes stressées, leur marge initiale et leur perte résiduelle au-delà de la marge (**SLOIM**).

**Pipeline cible :**
1. Définition et contrôle des portefeuilles des membres
2. Construction commune des scénarios FHS et stress
3. Calcul du PnL et de la marge initiale par membre
4. Calcul de la SLOIM par membre et par scénario
5. Dimensionnement du fonds de défaut selon Cover-2
6. Allocation des contributions entre les membres


## 0. Setup

In [38]:
import sys
from pathlib import Path

import pandas as pd

# Résolution du chemin racine du projet
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('ROOT =', ROOT)


ROOT = /home/jellilm/Desktop/Stage_QuantFactory/IM_pipeline./IM_pipeline


## 1. Définition des membres compensateurs

Ce prototype utilise six membres fictifs et des portefeuilles plus étoffés. Les positions sont construites à partir de l’univers obligataire disponible dans le pipeline. Dans une application réelle, elles seraient remplacées par les positions effectivement compensées par chaque membre.

- `Membre_A` porte principalement le risque court et moyen terme ;
- `Membre_B` est concentré sur le segment intermédiaire ;
- `Membre_C` et `Membre_D` sont fortement exposés aux maturités longues, avec des compositions différentes ;
- `Membre_E` présente un profil barbell, combinant court et long terme ;
- `Membre_F` est un portefeuille mixte comprenant plusieurs positions vendeuses.

La quantité est positive pour une position longue et négative pour une position courte.

In [39]:
members = {
    'Membre_A': [
        {'name': 'MOROCCO 2024 2.7% 19/01/26', 'maturity': 0.92, 'coupon_rate': 0.0270, 'nominal': 100.0, 'frequency': 1, 'quantity': 250000.0},
        {'name': 'MOROCCO 2014 5.6% 16/04/29', 'maturity': 4.16, 'coupon_rate': 0.0560, 'nominal': 100.0, 'frequency': 1, 'quantity': 180000.0},
        {'name': 'MOROCCO 2014 5.45% 06/08/29', 'maturity': 4.47, 'coupon_rate': 0.0545, 'nominal': 100.0, 'frequency': 1, 'quantity': 220000.0},
        {'name': 'MOROCCO 2022 2.4% 14/06/32', 'maturity': 7.33, 'coupon_rate': 0.0240, 'nominal': 100.0, 'frequency': 1, 'quantity': 80000.0},
        {'name': 'MOROCCO 2014 5.85% 31/03/34', 'maturity': 9.12, 'coupon_rate': 0.0585, 'nominal': 100.0, 'frequency': 1, 'quantity': -50000.0},
    ],
    'Membre_B': [
        {'name': 'MOROCCO 2014 5.6% 16/04/29', 'maturity': 4.16, 'coupon_rate': 0.0560, 'nominal': 100.0, 'frequency': 1, 'quantity': 120000.0},
        {'name': 'MOROCCO 2014 5.45% 06/08/29', 'maturity': 4.47, 'coupon_rate': 0.0545, 'nominal': 100.0, 'frequency': 1, 'quantity': -60000.0},
        {'name': 'MOROCCO 2022 2.4% 14/06/32', 'maturity': 7.33, 'coupon_rate': 0.0240, 'nominal': 100.0, 'frequency': 1, 'quantity': 240000.0},
        {'name': 'MOROCCO 2014 5.85% 31/03/34', 'maturity': 9.12, 'coupon_rate': 0.0585, 'nominal': 100.0, 'frequency': 1, 'quantity': 180000.0},
        {'name': 'MOROCCO 2021 3.45% 20/02/51', 'maturity': 26.02, 'coupon_rate': 0.0345, 'nominal': 100.0, 'frequency': 1, 'quantity': 25000.0},
    ],
    'Membre_C': [
        {'name': 'MOROCCO 2022 2.4% 14/06/32', 'maturity': 7.33, 'coupon_rate': 0.0240, 'nominal': 100.0, 'frequency': 1, 'quantity': 90000.0},
        {'name': 'MOROCCO 2014 5.85% 31/03/34', 'maturity': 9.12, 'coupon_rate': 0.0585, 'nominal': 100.0, 'frequency': 1, 'quantity': 130000.0},
        {'name': 'MOROCCO 2021 3.45% 20/02/51', 'maturity': 26.02, 'coupon_rate': 0.0345, 'nominal': 100.0, 'frequency': 1, 'quantity': 100000.0},
        {'name': 'MOROCCO 2024 4.9% 15/02/55', 'maturity': 30.01, 'coupon_rate': 0.0490, 'nominal': 100.0, 'frequency': 1, 'quantity': 220000.0},
        {'name': 'MOROCCO 2024 4 1/2% 19/04/55', 'maturity': 30.19, 'coupon_rate': 0.0450, 'nominal': 100.0, 'frequency': 1, 'quantity': 250000.0},
    ],
    'Membre_D': [
        {'name': 'MOROCCO 2014 5.45% 06/08/29', 'maturity': 4.47, 'coupon_rate': 0.0545, 'nominal': 100.0, 'frequency': 1, 'quantity': 70000.0},
        {'name': 'MOROCCO 2014 5.85% 31/03/34', 'maturity': 9.12, 'coupon_rate': 0.0585, 'nominal': 100.0, 'frequency': 1, 'quantity': 100000.0},
        {'name': 'MOROCCO 2021 3.45% 20/02/51', 'maturity': 26.02, 'coupon_rate': 0.0345, 'nominal': 100.0, 'frequency': 1, 'quantity': 140000.0},
        {'name': 'MOROCCO 2024 4.9% 15/02/55', 'maturity': 30.01, 'coupon_rate': 0.0490, 'nominal': 100.0, 'frequency': 1, 'quantity': 190000.0},
        {'name': 'MOROCCO 2024 4 1/2% 19/04/55', 'maturity': 30.19, 'coupon_rate': 0.0450, 'nominal': 100.0, 'frequency': 1, 'quantity': 210000.0},
    ],
    'Membre_E': [
        {'name': 'MOROCCO 2024 2.7% 19/01/26', 'maturity': 0.92, 'coupon_rate': 0.0270, 'nominal': 100.0, 'frequency': 1, 'quantity': 200000.0},
        {'name': 'MOROCCO 2014 5.6% 16/04/29', 'maturity': 4.16, 'coupon_rate': 0.0560, 'nominal': 100.0, 'frequency': 1, 'quantity': 100000.0},
        {'name': 'MOROCCO 2022 2.4% 14/06/32', 'maturity': 7.33, 'coupon_rate': 0.0240, 'nominal': 100.0, 'frequency': 1, 'quantity': -80000.0},
        {'name': 'MOROCCO 2021 3.45% 20/02/51', 'maturity': 26.02, 'coupon_rate': 0.0345, 'nominal': 100.0, 'frequency': 1, 'quantity': 120000.0},
        {'name': 'MOROCCO 2024 4.9% 15/02/55', 'maturity': 30.01, 'coupon_rate': 0.0490, 'nominal': 100.0, 'frequency': 1, 'quantity': 160000.0},
    ],
    'Membre_F': [
        {'name': 'MOROCCO 2024 2.7% 19/01/26', 'maturity': 0.92, 'coupon_rate': 0.0270, 'nominal': 100.0, 'frequency': 1, 'quantity': -150000.0},
        {'name': 'MOROCCO 2014 5.6% 16/04/29', 'maturity': 4.16, 'coupon_rate': 0.0560, 'nominal': 100.0, 'frequency': 1, 'quantity': 180000.0},
        {'name': 'MOROCCO 2014 5.45% 06/08/29', 'maturity': 4.47, 'coupon_rate': 0.0545, 'nominal': 100.0, 'frequency': 1, 'quantity': 130000.0},
        {'name': 'MOROCCO 2014 5.85% 31/03/34', 'maturity': 9.12, 'coupon_rate': 0.0585, 'nominal': 100.0, 'frequency': 1, 'quantity': -140000.0},
        {'name': 'MOROCCO 2021 3.45% 20/02/51', 'maturity': 26.02, 'coupon_rate': 0.0345, 'nominal': 100.0, 'frequency': 1, 'quantity': 60000.0},
        {'name': 'MOROCCO 2024 4 1/2% 19/04/55', 'maturity': 30.19, 'coupon_rate': 0.0450, 'nominal': 100.0, 'frequency': 1, 'quantity': 100000.0},
    ],
}

print(f'Nombre de membres : {len(members)}')
print(f'Nombre total de positions : {sum(len(portfolio) for portfolio in members.values())}')


Nombre de membres : 6
Nombre total de positions : 31


## 2. Contrôle des données membres

Avant de lancer un calcul de risque, on vérifie que le jeu contient au moins trois membres et que chaque position possède les champs nécessaires au moteur de valorisation.

In [40]:
required_fields = {
    'name', 'maturity', 'coupon_rate', 'nominal', 'frequency', 'quantity'
}

if len(members) < 3:
    raise ValueError('Le prototype Cover-2 doit contenir au moins trois membres.')

for member, portfolio in members.items():
    if not portfolio:
        raise ValueError(f'Le portefeuille de {member} est vide.')

    for position_number, position in enumerate(portfolio, start=1):
        missing_fields = required_fields.difference(position)
        if missing_fields:
            raise ValueError(
                f'{member}, position {position_number}: champs manquants {sorted(missing_fields)}'
            )
        if position['maturity'] <= 0 or position['nominal'] <= 0:
            raise ValueError(
                f'{member}, position {position_number}: maturité et nominal doivent être positifs.'
            )

print('Contrôles des portefeuilles : OK')


Contrôles des portefeuilles : OK


## 3. Synthèse des expositions par membre

Les montants ci-dessous sont des nominaux, pas encore des valeurs de marché. Ils permettent seulement de contrôler la composition des portefeuilles avant leur valorisation sur la courbe ZC.

In [41]:
summary_rows = []
position_rows = []

for member, portfolio in members.items():
    long_nominal = sum(
        position['quantity'] * position['nominal']
        for position in portfolio
        if position['quantity'] > 0
    )
    short_nominal = sum(
        position['quantity'] * position['nominal']
        for position in portfolio
        if position['quantity'] < 0
    )
    gross_nominal = sum(
        abs(position['quantity'] * position['nominal'])
        for position in portfolio
    )

    summary_rows.append({
        'member': member,
        'number_of_positions': len(portfolio),
        'long_nominal': long_nominal,
        'short_nominal': short_nominal,
        'net_nominal': long_nominal + short_nominal,
        'gross_nominal': gross_nominal,
    })

    for position in portfolio:
        position_rows.append({'member': member, **position})

member_summary = pd.DataFrame(summary_rows).set_index('member')
member_positions = pd.DataFrame(position_rows)

member_summary.style.format({
    'long_nominal': '{:,.0f}',
    'short_nominal': '{:,.0f}',
    'net_nominal': '{:,.0f}',
    'gross_nominal': '{:,.0f}',
})


,number_of_positions,long_nominal,short_nominal,net_nominal,gross_nominal
member,,,,,
Membre_A,5,"73,000,000","-5,000,000","68,000,000","78,000,000"
Membre_B,5,"56,500,000","-6,000,000","50,500,000","62,500,000"
Membre_C,5,"79,000,000",0,"79,000,000","79,000,000"
Membre_D,5,"71,000,000",0,"71,000,000","71,000,000"
Membre_E,5,"58,000,000","-8,000,000","50,000,000","66,000,000"
Membre_F,6,"47,000,000","-29,000,000","18,000,000","76,000,000"


In [42]:
member_positions[
    ['member', 'name', 'maturity', 'coupon_rate', 'nominal', 'frequency', 'quantity']
].sort_values(['member', 'maturity'])


,member,name,maturity,coupon_rate,nominal,frequency,quantity
0,Membre_A,MOROCCO 2024 2.7% 19/01/26,0.92,0.0270,100.0,1,250000.0
1,Membre_A,MOROCCO 2014 5.6% 16/04/29,4.16,0.0560,100.0,1,180000.0
2,Membre_A,MOROCCO 2014 5.45% 06/08/29,4.47,0.0545,100.0,1,220000.0
3,Membre_A,MOROCCO 2022 2.4% 14/06/32,7.33,0.0240,100.0,1,80000.0
4,Membre_A,MOROCCO 2014 5.85% 31/03/34,9.12,0.0585,100.0,1,-50000.0
5,Membre_B,MOROCCO 2014 5.6% 16/04/29,4.16,0.0560,100.0,1,120000.0
6,Membre_B,MOROCCO 2014 5.45% 06/08/29,4.47,0.0545,100.0,1,-60000.0
7,Membre_B,MOROCCO 2022 2.4% 14/06/32,7.33,0.0240,100.0,1,240000.0
8,Membre_B,MOROCCO 2014 5.85% 31/03/34,9.12,0.0585,100.0,1,180000.0
9,Membre_B,MOROCCO 2021 3.45% 20/02/51,26.02,0.0345,100.0,1,25000.0


## 4. Construction des scénarios communs

Le moteur de marge initiale est réutilisé sans changement. La courbe ZC, les returns, les volatilités EWMA et les scénarios sont construits une seule fois, puis appliqués à tous les membres.

Cette étape est essentielle : le Cover-2 doit comparer les membres sous le **même scénario de stress**.

In [43]:
from src.config import ModelConfig
from src.market_data.loader import load_zero_coupon_curve
from src.market_data.cleaner import clean_zero_coupon_curve
from src.risk.risk_factors import (
    build_zero_coupon_price_matrix,
    compute_historical_returns,
)
from src.risk.ewma import get_ewma_window, compute_ewma_volatility
from src.risk.scenarios import build_scaled_scenarios, build_unscaled_scenarios
from src.risk.pnl import (
    compute_portfolio_initial_value,
    compute_portfolio_pnl_under_scenarios,
)


In [44]:
config = ModelConfig(
    LP=2500,
    HP=5,
    SW=60,
    lambda_ewma=0.94,
    t0='2025-05-30',
    stress_start='2022-01-01',
    stress_end='2023-12-31',
    alpha=0.99,
    FHS_w=0.75,
    Stress_w=0.25,
    nominal=100.0,
)

config


ModelConfig(LP=2500, HP=5, SW=60, lambda_ewma=0.94, t0='2025-05-30', stress_start='2022-01-01', stress_end='2023-12-31', alpha=0.99, FHS_w=0.75, Stress_w=0.25, apc_buffer_pct=0.25, apc_floor_pct=0.25, metric='ES', nominal=100.0, model_version='1.1.0')

In [45]:
# Chargement et transformation de la courbe ZC
zc_curve_df = load_zero_coupon_curve(
    ROOT / 'data' / 'raw' / 'ZeroCouponCurve.csv'
)
zc_curve_df = clean_zero_coupon_curve(zc_curve_df)
zc_price_df = build_zero_coupon_price_matrix(
    zc_curve_df,
    nominal=config.nominal,
)
returns_df = compute_historical_returns(zc_price_df, HP=config.HP)
config.validate_against_data(returns_df)

# Volatilités EWMA
ewma_window = get_ewma_window(
    returns_df,
    t0=config.t0,
    LP=config.LP,
    SW=config.SW,
)
vol_df = compute_ewma_volatility(
    ewma_window,
    lambda_=config.lambda_ewma,
)

# Scénarios communs à tous les membres
scaled_df = build_scaled_scenarios(
    returns_df=returns_df,
    vol_df=vol_df,
    t0=config.t0,
    LP=config.LP,
)
unscaled_df = build_unscaled_scenarios(
    returns_df=returns_df,
    stress_start=config.stress_start,
    stress_end=config.stress_end,
)

current_zc_price_curve = zc_price_df.loc[pd.to_datetime(config.t0)]

print(f'Returns historiques       : {returns_df.shape}')
print(f'Scénarios FHS scalés      : {scaled_df.shape}')
print(f'Scénarios stress non scalés : {unscaled_df.shape}')


Returns historiques       : (3434, 33)
Scénarios FHS scalés      : (2500, 33)
Scénarios stress non scalés : (520, 33)


## 5. Calcul des PnL par membre

Chaque portefeuille est revalorisé sous chaque scénario avec la méthode Full Revaluation :

$$PnL(m,s)=V^{(s)}(m)-V(t_0,m).$$

Les lignes des matrices correspondent aux scénarios et les colonnes aux membres.

In [46]:
initial_values = {}
pnl_fhs = {}
pnl_stress = {}

for member, portfolio in members.items():
    initial_values[member] = compute_portfolio_initial_value(
        current_zc_price_curve=current_zc_price_curve,
        portfolio=portfolio,
        zc_nominal=config.nominal,
    )
    pnl_fhs[member] = compute_portfolio_pnl_under_scenarios(
        current_zc_price_curve=current_zc_price_curve,
        scenario_returns_df=scaled_df,
        portfolio=portfolio,
        zc_nominal=config.nominal,
    )
    pnl_stress[member] = compute_portfolio_pnl_under_scenarios(
        current_zc_price_curve=current_zc_price_curve,
        scenario_returns_df=unscaled_df,
        portfolio=portfolio,
        zc_nominal=config.nominal,
    )

initial_values_by_member = pd.Series(initial_values, name='initial_value')
pnl_fhs_by_member = pd.DataFrame(pnl_fhs)
pnl_stress_by_member = pd.DataFrame(pnl_stress)

print(f'Matrice PnL FHS    : {pnl_fhs_by_member.shape}')
print(f'Matrice PnL stress : {pnl_stress_by_member.shape}')

initial_values_by_member.to_frame().style.format('{:,.2f}')


Matrice PnL FHS    : (2500, 6)
Matrice PnL stress : (520, 6)


,initial_value
Membre_A,"73,224,809.87"
Membre_B,"56,582,051.52"
Membre_C,"93,127,451.19"
Membre_D,"83,667,061.32"
Membre_E,"56,217,722.92"
Membre_F,"20,149,048.35"


In [47]:
pnl_stress_by_member.head().style.format('{:,.2f}')


,Membre_A,Membre_B,Membre_C,Membre_D,Membre_E,Membre_F
date,,,,,,
2022-01-03 00:00:00,"29,786.59","23,374.24","105,860.49","99,221.58","58,247.30","27,994.65"
2022-01-04 00:00:00,"40,095.87","22,351.55","47,921.61","48,973.33","36,145.80","21,438.95"
2022-01-05 00:00:00,"19,020.23","19,110.98","75,616.13","68,204.75","41,078.70","11,698.24"
2022-01-06 00:00:00,"15,442.71","20,858.04","78,992.11","70,921.82","39,284.51","13,368.68"
2022-01-07 00:00:00,"45,384.29","21,638.38","-7,644.13","-3,304.32","8,578.25","9,013.47"


## 6. Conversion des PnL en pertes stressées

Un PnL négatif représente une perte. Les gains sont ramenés à zéro :

$$L(m,s)=\max\left(-PnL(m,s),0\right).$$

La matrice obtenue sera utilisée pour calculer la perte résiduelle SLOIM.

In [48]:
stress_losses_by_member = (-pnl_stress_by_member).clip(lower=0.0)

stress_loss_summary = pd.DataFrame({
    'initial_value': initial_values_by_member,
    'worst_stress_loss': stress_losses_by_member.max(),
    'average_stress_loss': stress_losses_by_member.mean(),
    'loss_scenarios': (stress_losses_by_member > 0).sum(),
})
stress_loss_summary['worst_loss_ratio'] = (
    stress_loss_summary['worst_stress_loss']
    / stress_loss_summary['initial_value'].abs()
)

stress_loss_summary.style.format({
    'initial_value': '{:,.2f}',
    'worst_stress_loss': '{:,.2f}',
    'average_stress_loss': '{:,.2f}',
    'worst_loss_ratio': '{:.2%}',
})


,initial_value,worst_stress_loss,average_stress_loss,loss_scenarios,worst_loss_ratio
Membre_A,"73,224,809.87","1,183,369.55","58,832.93",145,1.62%
Membre_B,"56,582,051.52","1,630,169.92","52,622.05",133,2.88%
Membre_C,"93,127,451.19","4,773,745.69","241,744.76",206,5.13%
Membre_D,"83,667,061.32","4,479,034.09","225,080.06",210,5.35%
Membre_E,"56,217,722.92","2,291,051.84","121,649.39",204,4.08%
Membre_F,"20,149,048.35","1,365,833.96","90,377.70",211,6.78%


## 7. Marge initiale individuelle par membre

La marge initiale est calculée séparément pour chaque membre avec le même modèle hybride que dans le pipeline IM :

$$IM(m)=w_{FHS}\,ES_{FHS}(m)+w_{stress}\,ES_{stress}(m).$$

L’IM reste une ressource individuelle : chaque membre apporte une marge déterminée à partir du risque de son propre portefeuille.

In [49]:
from src.risk.es import compute_es_from_pnl


In [50]:
im_rows = []

for member in members:
    es_fhs = compute_es_from_pnl(
        pnl_fhs_by_member[member],
        alpha=config.alpha,
    )
    es_stress = compute_es_from_pnl(
        pnl_stress_by_member[member],
        alpha=config.alpha,
    )
    initial_margin = (
        config.FHS_w * es_fhs
        + config.Stress_w * es_stress
    )

    im_rows.append({
        'member': member,
        'ES_FHS': es_fhs,
        'ES_stress': es_stress,
        'initial_margin': initial_margin,
    })

im_by_member = pd.DataFrame(im_rows).set_index('member')

im_by_member.style.format({
    'ES_FHS': '{:,.2f}',
    'ES_stress': '{:,.2f}',
    'initial_margin': '{:,.2f}',
})


,ES_FHS,ES_stress,initial_margin
member,,,
Membre_A,"1,252,591.35","1,153,666.01","1,227,860.01"
Membre_B,"1,846,804.61","1,337,355.40","1,719,442.31"
Membre_C,"2,433,596.58","4,253,737.50","2,888,631.81"
Membre_D,"1,995,127.29","3,978,918.93","2,491,075.20"
Membre_E,"1,064,932.68","2,026,941.46","1,305,434.88"
Membre_F,"812,359.25","1,209,371.41","911,612.29"


## 8. Perte résiduelle au-delà de la marge — SLOIM

Pour chaque membre $m$ et chaque scénario de stress $s$, on retranche la marge initiale à la perte stressée :

$$SLOIM(m,s)=\max\left(0,L(m,s)-IM(m)\right).$$

- si la perte est inférieure à l’IM, la SLOIM vaut zéro ;
- si la perte dépasse l’IM, la SLOIM représente la partie qui n’est plus couverte par la marge du membre.

C’est cette perte résiduelle, et non la perte stressée totale, qui doit être prise en charge par les ressources suivantes de la cascade de défaut.

In [51]:
initial_margin_by_member = im_by_member['initial_margin']

sloim_by_member = stress_losses_by_member.sub(
    initial_margin_by_member,
    axis='columns',
).clip(lower=0.0)

sloim_summary = pd.DataFrame({
    'initial_margin': initial_margin_by_member,
    'worst_stress_loss': stress_losses_by_member.max(),
    'worst_sloim': sloim_by_member.max(),
    'sloim_scenarios': (sloim_by_member > 0).sum(),
})
sloim_summary['maximum_loss_covered_by_im'] = (
    sloim_summary['worst_stress_loss']
    - sloim_summary['worst_sloim']
)

sloim_summary.style.format({
    'initial_margin': '{:,.2f}',
    'worst_stress_loss': '{:,.2f}',
    'worst_sloim': '{:,.2f}',
    'maximum_loss_covered_by_im': '{:,.2f}',
})


,initial_margin,worst_stress_loss,worst_sloim,sloim_scenarios,maximum_loss_covered_by_im
Membre_A,"1,227,860.01","1,183,369.55",0.00,0,"1,183,369.55"
Membre_B,"1,719,442.31","1,630,169.92",0.00,0,"1,630,169.92"
Membre_C,"2,888,631.81","4,773,745.69","1,885,113.88",10,"2,888,631.81"
Membre_D,"2,491,075.20","4,479,034.09","1,987,958.89",10,"2,491,075.20"
Membre_E,"1,305,434.88","2,291,051.84","985,616.96",10,"1,305,434.88"
Membre_F,"911,612.29","1,365,833.96","454,221.67",10,"911,612.29"


## 9. Dimensionnement du fonds de défaut — Cover-2

Pour chaque scénario de stress, on classe les SLOIM des membres de la plus grande à la plus petite. Le besoin Cover-2 du scénario est la somme des deux plus grandes pertes résiduelles :

$$Cover2(s)=SLOIM_{(1)}(s)+SLOIM_{(2)}(s).$$

Les deux défauts sont évalués sous le **même scénario**. Le couple de membres retenu peut donc changer selon le scénario.

Le fonds de défaut est le maximum du besoin Cover-2 sur tous les scénarios :

$$DF_{Cover2}=\max_s Cover2(s).$$

In [52]:
if sloim_by_member.shape[1] < 2:
    raise ValueError('Le calcul Cover-2 nécessite au moins deux membres.')

cover2_rows = []

for scenario, member_sloim in sloim_by_member.iterrows():
    ranking = member_sloim.sort_values(ascending=False)

    first_member = ranking.index[0]
    second_member = ranking.index[1]
    first_sloim = float(ranking.iloc[0])
    second_sloim = float(ranking.iloc[1])

    cover2_rows.append({
        'scenario': scenario,
        'first_member': first_member,
        'first_sloim': first_sloim,
        'second_member': second_member,
        'second_sloim': second_sloim,
        'cover2_requirement': first_sloim + second_sloim,
    })

cover2_by_scenario = pd.DataFrame(cover2_rows).set_index('scenario')

cover2_by_scenario.nlargest(10, 'cover2_requirement').style.format({
    'first_sloim': '{:,.2f}',
    'second_sloim': '{:,.2f}',
    'cover2_requirement': '{:,.2f}',
})


,first_member,first_sloim,second_member,second_sloim,cover2_requirement
scenario,,,,,
2023-01-10 00:00:00,Membre_D,"1,987,958.89",Membre_C,"1,885,113.88","3,873,072.77"
2023-01-05 00:00:00,Membre_D,"1,842,135.76",Membre_C,"1,711,240.38","3,553,376.14"
2023-01-13 00:00:00,Membre_D,"1,809,769.34",Membre_C,"1,724,558.44","3,534,327.78"
2022-12-23 00:00:00,Membre_D,"1,602,633.48",Membre_C,"1,479,346.80","3,081,980.28"
2023-01-09 00:00:00,Membre_D,"1,198,787.00",Membre_C,"1,099,796.58","2,298,583.57"
2022-10-03 00:00:00,Membre_D,"485,777.95",Membre_C,"290,578.07","776,356.02"
2022-10-06 00:00:00,Membre_D,"421,149.91",Membre_C,"212,990.44","634,140.35"
2022-10-07 00:00:00,Membre_D,"420,796.75",Membre_C,"212,665.83","633,462.58"
2022-12-27 00:00:00,Membre_D,"310,978.31",Membre_C,"180,098.61","491,076.92"


### Scénario contraignant et montant du fonds

Le scénario contraignant est celui qui maximise la somme des deux plus grandes SLOIM. Le tableau suivant détaille les pertes, les marges et les SLOIM des membres sous ce scénario.

In [53]:
binding_scenario = cover2_by_scenario['cover2_requirement'].idxmax()
default_fund_cover2 = float(
    cover2_by_scenario.loc[binding_scenario, 'cover2_requirement']
)
binding_cover2 = cover2_by_scenario.loc[binding_scenario]

binding_scenario_detail = pd.DataFrame({
    'stress_loss': stress_losses_by_member.loc[binding_scenario],
    'initial_margin': initial_margin_by_member,
    'sloim': sloim_by_member.loc[binding_scenario],
}).sort_values('sloim', ascending=False)

print(f'Scénario contraignant : {binding_scenario}')
print(f'Premier membre        : {binding_cover2["first_member"]}')
print(f'Deuxième membre       : {binding_cover2["second_member"]}')
print(f'Fonds Cover-2         : {default_fund_cover2:,.2f}')

binding_scenario_detail.style.format({
    'stress_loss': '{:,.2f}',
    'initial_margin': '{:,.2f}',
    'sloim': '{:,.2f}',
})


Scénario contraignant : 2023-01-10 00:00:00
Premier membre        : Membre_D
Deuxième membre       : Membre_C
Fonds Cover-2         : 3,873,072.77


,stress_loss,initial_margin,sloim
Membre_D,"4,479,034.09","2,491,075.20","1,987,958.89"
Membre_C,"4,773,745.69","2,888,631.81","1,885,113.88"
Membre_E,"2,291,051.84","1,305,434.88","985,616.96"
Membre_F,"1,365,833.96","911,612.29","454,221.67"
Membre_A,"126,636.83","1,227,860.01",0.00
Membre_B,"200,364.78","1,719,442.31",0.00


In [54]:
expected_cover2 = sloim_by_member.loc[binding_scenario].nlargest(2).sum()

if abs(default_fund_cover2 - expected_cover2) > 1e-9:
    raise AssertionError('Le Cover-2 ne correspond pas aux deux plus grandes SLOIM.')
if default_fund_cover2 < 0:
    raise AssertionError('Le fonds de défaut ne peut pas être négatif.')

print('Contrôles Cover-2 : OK')


Contrôles Cover-2 : OK


## 10. Allocation des contributions entre les membres

Conformément à la note méthodologique, le fonds est réparti entre les membres selon une clé de risque, avec un minimum par membre, puis un buffer prudentiel.

Deux clés sont comparées :

- **prorata de la marge initiale** : méthode stable, simple et peu sensible à un scénario isolé ;
- **prorata de la pire SLOIM** : méthode plus directement liée au risque de queue de chaque membre.

La calibration ci-dessous est illustrative, car la note ne fixe pas les valeurs numériques :

- buffer prudentiel : 10 % du Cover-2 ;
- contribution minimale : 2 % du fonds bufferisé pour chaque membre.

Après attribution des minima, le solde du fonds est réparti au prorata de la clé de risque choisie.

In [55]:
prudential_buffer_pct = 0.10
minimum_share_per_member = 0.02
published_allocation_method = 'initial_margin'

number_of_members = len(members)
if minimum_share_per_member < 0:
    raise ValueError('La part minimale doit être positive ou nulle.')
if minimum_share_per_member * number_of_members >= 1.0:
    raise ValueError(
        'La somme des minima doit rester strictement inférieure à 100 % du fonds.'
    )
if prudential_buffer_pct < 0:
    raise ValueError('Le buffer prudentiel ne peut pas être négatif.')

default_fund_with_buffer = default_fund_cover2 * (1.0 + prudential_buffer_pct)

print(f'Cover-2 brut            : {default_fund_cover2:,.2f}')
print(f'Buffer prudentiel       : {prudential_buffer_pct:.1%}')
print(f'Fonds après buffer      : {default_fund_with_buffer:,.2f}')
print(f'Minimum par membre      : {minimum_share_per_member:.1%}')


Cover-2 brut            : 3,873,072.77
Buffer prudentiel       : 10.0%
Fonds après buffer      : 4,260,380.05
Minimum par membre      : 2.0%


In [56]:
def allocate_default_fund(
    total_fund,
    risk_basis,
    minimum_share=0.0,
):
    """Répartit le fonds avec un minimum égal par membre puis au prorata du risque."""
    basis = pd.Series(risk_basis, dtype=float).clip(lower=0.0)
    if basis.empty:
        raise ValueError('La base d’allocation est vide.')
    if basis.sum() <= 0:
        raise ValueError('La base d’allocation doit contenir au moins une valeur positive.')
    if minimum_share * len(basis) >= 1.0:
        raise ValueError('La somme des minima dépasse le fonds disponible.')

    minimum_amount = total_fund * minimum_share
    residual_pool = total_fund - minimum_amount * len(basis)
    risk_weight = basis / basis.sum()
    contribution = minimum_amount + residual_pool * risk_weight

    return pd.DataFrame({
        'risk_basis': basis,
        'risk_weight': risk_weight,
        'minimum_contribution': minimum_amount,
        'risk_based_contribution': residual_pool * risk_weight,
        'total_contribution': contribution,
        'final_share': contribution / total_fund,
    })


### Comparaison des deux clés d’allocation

Pour la méthode SLOIM, la base de risque retenue est la **pire SLOIM historique de chaque membre** sur l’ensemble des scénarios de stress.

In [57]:
allocation_by_im = allocate_default_fund(
    total_fund=default_fund_with_buffer,
    risk_basis=initial_margin_by_member,
    minimum_share=minimum_share_per_member,
)

worst_sloim_by_member = sloim_by_member.max()
allocation_by_sloim = allocate_default_fund(
    total_fund=default_fund_with_buffer,
    risk_basis=worst_sloim_by_member,
    minimum_share=minimum_share_per_member,
)

allocation_comparison = pd.DataFrame({
    'initial_margin': initial_margin_by_member,
    'worst_sloim': worst_sloim_by_member,
    'contribution_by_im': allocation_by_im['total_contribution'],
    'share_by_im': allocation_by_im['final_share'],
    'contribution_by_sloim': allocation_by_sloim['total_contribution'],
    'share_by_sloim': allocation_by_sloim['final_share'],
})

allocation_comparison.style.format({
    'initial_margin': '{:,.2f}',
    'worst_sloim': '{:,.2f}',
    'contribution_by_im': '{:,.2f}',
    'share_by_im': '{:.2%}',
    'contribution_by_sloim': '{:,.2f}',
    'share_by_sloim': '{:.2%}',
})


,initial_margin,worst_sloim,contribution_by_im,share_by_im,contribution_by_sloim,share_by_sloim
Membre_A,"1,227,860.01",0.00,"521,795.95",12.25%,"85,207.60",2.00%
Membre_B,"1,719,442.31",0.00,"696,587.14",16.35%,"85,207.60",2.00%
Membre_C,"2,888,631.81","1,885,113.88","1,112,314.11",26.11%,"1,415,466.07",33.22%
Membre_D,"2,491,075.20","1,987,958.89","970,955.49",22.79%,"1,488,040.17",34.93%
Membre_E,"1,305,434.88","985,616.96","549,379.13",12.90%,"780,722.77",18.33%
Membre_F,"911,612.29","454,221.67","409,348.22",9.61%,"405,735.84",9.52%


### Allocation retenue pour le prototype

Le prototype retient par défaut l’allocation au prorata de la marge initiale. Ce choix donne une clé plus stable et garantit que chaque membre contribue, même lorsque sa SLOIM historique est nulle ou très rare. La méthode peut être remplacée par `sloim` dans le paramètre `published_allocation_method`.

In [58]:
allocation_methods = {
    'initial_margin': allocation_by_im,
    'sloim': allocation_by_sloim,
}
if published_allocation_method not in allocation_methods:
    raise ValueError(
        "published_allocation_method doit valoir 'initial_margin' ou 'sloim'."
    )

member_contributions = allocation_methods[published_allocation_method].copy()
member_contributions.index.name = 'member'

print(f'Méthode retenue : {published_allocation_method}')
member_contributions.style.format({
    'risk_basis': '{:,.2f}',
    'risk_weight': '{:.2%}',
    'minimum_contribution': '{:,.2f}',
    'risk_based_contribution': '{:,.2f}',
    'total_contribution': '{:,.2f}',
    'final_share': '{:.2%}',
})


Méthode retenue : initial_margin


,risk_basis,risk_weight,minimum_contribution,risk_based_contribution,total_contribution,final_share
member,,,,,,
Membre_A,"1,227,860.01",11.65%,"85,207.60","436,588.35","521,795.95",12.25%
Membre_B,"1,719,442.31",16.31%,"85,207.60","611,379.54","696,587.14",16.35%
Membre_C,"2,888,631.81",27.40%,"85,207.60","1,027,106.50","1,112,314.11",26.11%
Membre_D,"2,491,075.20",23.63%,"85,207.60","885,747.89","970,955.49",22.79%
Membre_E,"1,305,434.88",12.38%,"85,207.60","464,171.53","549,379.13",12.90%
Membre_F,"911,612.29",8.65%,"85,207.60","324,140.62","409,348.22",9.61%


In [59]:
allocated_total = float(member_contributions['total_contribution'].sum())
minimum_amount = default_fund_with_buffer * minimum_share_per_member

if abs(allocated_total - default_fund_with_buffer) > 1e-6:
    raise AssertionError('Les contributions ne somment pas au fonds bufferisé.')
if (member_contributions['total_contribution'] < minimum_amount - 1e-9).any():
    raise AssertionError('Au moins un membre ne respecte pas la contribution minimale.')
if abs(member_contributions['final_share'].sum() - 1.0) > 1e-12:
    raise AssertionError('Les parts finales ne somment pas à 100 %.')

print(f'Total alloué : {allocated_total:,.2f}')
print('Contrôles d’allocation : OK')


Total alloué : 4,260,380.05
Contrôles d’allocation : OK


## Conclusion

Le pipeline couvre désormais le calcul complet du fonds de défaut : pertes stressées, marge initiale individuelle, SLOIM, dimensionnement Cover-2, buffer prudentiel et allocation des contributions entre les membres.

Les paramètres de minimum, de buffer et de clé d’allocation devront être calibrés et validés par la politique de risque de la CCP avant une utilisation de production.